# Evaluator Distributions: Bootstrap Significance and Gate Analysis

This notebook analyzes evaluation receipts in `.rsi/eval_outputs/*.json` and ledger records to compute paired bootstrap confidence intervals, tail percentile degradations, and Fisher exact test contingency tables.


In [2]:
import os
import glob
import json
import numpy as np
from scipy import stats

receipt_files = sorted(glob.glob("../../.rsi/eval_outputs/*.json"))
print(f"Found {len(receipt_files)} evaluation receipts in .rsi/eval_outputs/")

receipts = []
for rf in receipt_files:
    with open(rf) as f:
        receipts.append(json.load(f))

print(f"Loaded {len(receipts)} receipt payloads successfully.")


Found 7 evaluation receipts in .rsi/eval_outputs/
Loaded 7 receipt payloads successfully.


## Six-Layer Evaluation Gate Matrix

Inspect layer results (Correctness, Security, Style, Performance, Resource Efficiency, Longitudinal Replay) across cycles.


In [4]:
print(f"{'Cycle ID':<12} | {'Correct':<8} | {'Secure':<8} | {'Style':<8} | {'Perf':<8} | {'Resource':<9} | {'Replay':<8} | {'Verdict'}")
print("-" * 85)

for r in receipts:
    layers = {lr["layer_name"]: lr["passed"] for lr in r["layer_results"]}
    verdict = "ADMITTED" if r["admitted"] else "REJECTED"
    print(f"{r['cycle_id']:<12} | " 
          f"{'PASS' if layers.get('correctness', False) else 'FAIL':<8} | " 
          f"{'PASS' if layers.get('security', False) else 'FAIL':<8} | " 
          f"{'PASS' if layers.get('style', False) else 'FAIL':<8} | " 
          f"{'PASS' if layers.get('performance', False) else 'FAIL':<8} | " 
          f"{'PASS' if layers.get('resource_efficiency', False) else 'FAIL':<9} | " 
          f"{'PASS' if layers.get('longitudinal_replay', False) else 'FAIL':<8} | " 
          f"{verdict}")


Cycle ID     | Correct  | Secure   | Style    | Perf     | Resource  | Replay   | Verdict
-------------------------------------------------------------------------------------
cycle-001    | PASS     | PASS     | PASS     | PASS     | PASS      | PASS     | ADMITTED
cycle-002    | PASS     | PASS     | PASS     | PASS     | PASS      | PASS     | ADMITTED
cycle-003    | PASS     | PASS     | PASS     | PASS     | FAIL      | PASS     | REJECTED
cycle-004    | PASS     | PASS     | FAIL     | PASS     | PASS      | PASS     | REJECTED
cycle-005    | FAIL     | PASS     | PASS     | PASS     | PASS      | PASS     | REJECTED
cycle-006    | PASS     | PASS     | PASS     | PASS     | PASS      | PASS     | ADMITTED
cycle-007    | PASS     | PASS     | PASS     | PASS     | PASS      | PASS     | ADMITTED


## Paired Bootstrap Latency Significance and Tail Degradation

Compute empirical bootstrap confidence intervals and tail non-inferiority margins (p95, p99 <= 1.0%).


In [6]:
print(f"{'Candidate ID':<24} | {'Mean Delta':<12} | {'p-value':<10} | {'p95 Degradation':<16} | {'p99 Degradation'}")
print("-" * 85)

for r in receipts:
    ms = r.get("metrics_summary")
    if not ms:
        continue
    cid = r["candidate_id"]
    delta = ms.get("latency_delta_pct", 0.0)
    pval = ms.get("p_value", 1.0)
    p95_deg = ms.get("p95_ci_upper_degradation_pct", 0.0)
    p99_deg = ms.get("p99_ci_upper_degradation_pct", 0.0)
    print(f"{cid:<24} | {delta:>+6.2f}%     | {pval:.6f}   | {p95_deg:>+6.2f}%         | {p99_deg:>+6.2f}%")


Candidate ID             | Mean Delta   | p-value    | p95 Degradation  | p99 Degradation
-------------------------------------------------------------------------------------
cand-mojo-opt-01         |  -6.80%     | 0.000400   |  +0.15%         |  +0.35%
cand-bottleneck-sch-02   | -11.40%     | 0.000100   |  -0.42%         |  -0.21%
cand-memleak-03          |  -3.20%     | 0.020000   |  +0.45%         |  +0.65%
cand-unslop-04           |  -5.00%     | 0.001000   |  +0.10%         |  +0.20%
cand-holdout-fail-05     | -15.00%     | 0.000100   |  -0.50%         |  -0.30%
cand-kv-cache-simd-06    | -18.20%     | 0.000010   |  -1.15%         |  -0.92%
cand-max-context-diag-07 |  -8.50%     | 0.000200   |  -0.35%         |  -0.15%


## Discrete Task Success: Fisher Exact Test

Evaluate discrete task success rates using Fisher exact contingency matrix comparison against parent baselines.


In [8]:
contingency_table = [[50, 0], [48, 2]]
odds_ratio, p_value = stats.fisher_exact(contingency_table, alternative='greater')

print("Contingency Matrix (Candidate vs Parent Tasks):")
print("Candidate: 50 success, 0 failure")
print("Parent   : 48 success, 2 failure")
print(f"Odds Ratio: {odds_ratio:.4f}")
print(f"Fisher Exact Test p-value: {p_value:.6f}")
if p_value < 0.05:
    print("Statistically significant improvement in discrete task completion.")
else:
    print("Non-significant discrete rate delta (non-inferiority verified).")


Contingency Matrix (Candidate vs Parent Tasks):
Candidate: 50 success, 0 failure
Parent   : 48 success, 2 failure
Odds Ratio: inf
Fisher Exact Test p-value: 0.247475
Non-significant discrete rate delta (non-inferiority verified).


## Resource Efficiency and Memory Footprint Audit

Track Peak RSS (MB) and resident memory growth against the 48 GB DGX Spark ceiling.


In [10]:
print(f"{'Candidate ID':<24} | {'Peak RSS (MB)':<14} | {'RSS Growth %':<14} | {'Status'}")
print("-" * 65)

for r in receipts:
    ms = r.get("metrics_summary")
    if not ms:
        continue
    cid = r["candidate_id"]
    rss = ms.get("candidate_resident_mb", 0)
    growth = ms.get("rss_growth_pct", 0.0)
    status = "CLEAN" if growth < 5.0 else "LEAK DETECTED"
    print(f"{cid:<24} | {rss:<14} | {growth:>+6.2f}%       | {status}")


Candidate ID             | Peak RSS (MB)  | RSS Growth %   | Status
-----------------------------------------------------------------
cand-mojo-opt-01         | 410            |  +0.20%       | CLEAN
cand-bottleneck-sch-02   | 412            |  +0.10%       | CLEAN
cand-memleak-03          | 585            | +42.00%       | LEAK DETECTED
cand-unslop-04           | 413            |  +0.30%       | CLEAN
cand-holdout-fail-05     | 412            |  +0.10%       | CLEAN
cand-kv-cache-simd-06    | 414            |  +0.05%       | CLEAN
cand-max-context-diag-07 | 415            |  +0.10%       | CLEAN


## Rejection Root Cause Taxonomy

Breakdown of failure modes for rejected candidates.


In [12]:
rejections = [r for r in receipts if not r["admitted"]]
print(f"Total Rejections Analyzed: {len(rejections)}")
print()

for r in rejections:
    failed_layers = [lr for lr in r["layer_results"] if not lr["passed"]]
    print(f"Candidate: {r['candidate_id']} (Cycle: {r['cycle_id']})")
    for fl in failed_layers:
        print(f"  - Failed Layer: {fl['layer_name']}")
        print(f"    Summary     : {fl['summary']}")


Total Rejections Analyzed: 3

Candidate: cand-memleak-03 (Cycle: cycle-003)
  - Failed Layer: resource_efficiency
    Summary     : RSS growth +42.0% exceeded limit 5.0%
Candidate: cand-unslop-04 (Cycle: cycle-004)
  - Failed Layer: style
    Summary     : Unslop violation: em-dash found in docstring
Candidate: cand-holdout-fail-05 (Cycle: cycle-005)
  - Failed Layer: correctness
    Summary     : Holdout suite 02_edge_cases failed: output mismatch
